# 🎬 AI Auto Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saigonvet8312-coder/AI_auto_video/blob/main/AI_auto_video.ipynb)

Kịch bản → giọng đọc → hình ảnh động → video dọc 9:16.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`.
Chạy **Phần 1 → 2 → 3**. Từ lần sau chỉ cần chạy **Phần 2 → 3** (nếu chưa mất phiên).

## Phần 1 — Cài đặt *(một lần mỗi phiên)*

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq ffmpeg fonts-noto fonts-noto-color-emoji fonts-liberation >/dev/null
pip install -q "gradio>=5,<6" omnivoice playwright numpy pillow requests anthropic deep-translator
python -m playwright install --with-deps chromium >/dev/null
fc-cache -f >/dev/null
echo "✅ Cài đặt xong"

## Phần 2 — Mã nguồn & thư mục dự án

In [ ]:
#@title Cấu hình { display-mode: "form" }
USE_DRIVE = True #@param {type:"boolean"}
VOICES_DIR = "/content/drive/MyDrive/AI_auto_video_voices" #@param {type:"string"}
REPO_URL = "https://github.com/saigonvet8312-coder/AI_auto_video.git" #@param {type:"string"}

import glob, os, subprocess
REPO_DIR = "/content/AI_auto_video"

# Dự án chỉ lưu tạm trong phiên Colab (xong thì tải về). Chỉ thư viện giọng đọc được lưu trên Drive.
PROJECTS_DIR = "/content/projects"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    VOICES_DIR = "/content/voices"
os.makedirs(PROJECTS_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Nếu repo không có thư mục app/ (các file .py nằm thẳng ở gốc), tự gom vào app/
import shutil
MODULES = ["__init__", "aiimage", "assemble", "autobg", "director", "jobs", "main", "media", "project",
           "render", "scriptkit", "subjects", "templates", "tts", "vnnum", "voices"]
tracked_app = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--error-unmatch", "app/main.py"],
                             capture_output=True).returncode == 0
if not tracked_app:
    missing = [m for m in MODULES if not os.path.exists(f"{REPO_DIR}/{m}.py")]
    if not missing:
        os.makedirs(f"{REPO_DIR}/app", exist_ok=True)
        for m in MODULES:
            shutil.copy(f"{REPO_DIR}/{m}.py", f"{REPO_DIR}/app/{m}.py")
        print("ℹ️ Đã gom các file .py vào thư mục app/")
    elif len(missing) < len(MODULES):
        raise RuntimeError("Repo còn thiếu file: " + ", ".join(m + ".py" for m in missing) +
                           ". Hãy upload đủ các file rồi chạy lại.")

# Tìm thư mục chứa app/main.py (kể cả khi repo bị lồng thêm một cấp thư mục)
hits = sorted(glob.glob(f"{REPO_DIR}/**/app/main.py", recursive=True), key=len)
if not hits:
    listing = subprocess.run(["ls", "-la", REPO_DIR], capture_output=True, text=True).stdout
    raise RuntimeError(
        "Repo chưa có app/main.py. Hãy upload cả thư mục app/ lên GitHub rồi chạy lại.\n"
        f"Nội dung repo hiện tại:\n{listing}"
    )
APP_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(APP_ROOT)
os.environ["PYTHONPATH"] = APP_ROOT
os.environ["AVG_PROJECTS_DIR"] = PROJECTS_DIR
os.environ["AVG_VOICES_DIR"] = VOICES_DIR
print("✅ Mã nguồn:", APP_ROOT)
print("🎙️ Thư viện giọng:", VOICES_DIR)

## Phần 3 — Mở giao diện
Chạy cell dưới, đợi hiện dòng **Running on public URL**, bấm link để mở ở tab mới. Cell này sẽ chạy liên tục khi giao diện còn mở.

In [ ]:
!cd "{APP_ROOT}" && python -m app.main

## Ghi chú
- Mô hình giọng đọc tải lần đầu mất vài phút (khi bấm “Tạo giọng” lần đầu).
- **Dự án không được lưu lại** sau phiên Colab: khi video xong, hãy tải `ket-qua.zip` về ngay.
- **Giọng đọc** được lưu trên Google Drive (thư mục `AI_auto_video_voices`): lưu một lần ở tab ②, lần sau chỉ cần chọn trong danh sách.
- Cập nhật mã: chạy lại Phần 2 rồi Phần 3.
